In [1]:
import pandas as pd

ops = pd.read_csv('operation_history.csv')
docs = pd.read_csv('document.csv')

ops['date_created'] = pd.to_datetime(ops['date_created'], errors='coerce')
docs['date_created'] = pd.to_datetime(docs['date_created'], errors='coerce')
docs['date_closed'] = pd.to_datetime(docs['date_closed'], errors='coerce')

In [2]:
delivery_rows = ops[
    (ops['document_type'] == 'DELIVERY') &
    (ops['operation_type'] == 'DOCUMENT_ADD_CARGO_PLACE') &
    (ops['current_office_uuid'].notna())
].copy()
delivery_rows = delivery_rows.sort_values(['cargo_place_uuid', 'date_created', 'uuid'])

receiver_office = (
    delivery_rows.groupby('cargo_place_uuid', as_index=False)
    .first()[['cargo_place_uuid', 'current_office_uuid']]
    .rename(columns={'current_office_uuid': 'receiver_office_uuid'})
)

In [3]:
ops_in_receiver = ops.merge(receiver_office, on='cargo_place_uuid', how='inner')
ops_in_receiver = ops_in_receiver[
    (ops_in_receiver['current_office_uuid'] == ops_in_receiver['receiver_office_uuid']) &
    (ops_in_receiver['operation_type'] == 'DOCUMENT_ADD_CARGO_PLACE')
].copy()

ops_in_receiver = ops_in_receiver.sort_values(['cargo_place_uuid', 'date_created', 'uuid'])

first_doc = (
    ops_in_receiver.groupby('cargo_place_uuid', as_index=False)
    .first()[['cargo_place_uuid', 'receiver_office_uuid', 'document_uuid', 'date_created']]
    .rename(columns={'date_created': 'first_seen_at_receiver_dt'})
)

In [4]:
task1_result = first_doc.merge(
    docs[['uuid', 'number']],
    left_on='document_uuid',
    right_on='uuid',
    how='left'
)

task1_result = task1_result[['receiver_office_uuid', 'cargo_place_uuid', 'number']].rename(
    columns={'number': 'document_number'})

task1_result.to_csv('result.csv', index=False)

task1_result

,receiver_office_uuid,cargo_place_uuid,document_number
0,b6ec1cbf-9889-47a0-85ed-de079a1064d9,529c9def-c5d1-44f0-9d20-674e0ac5774a,CN/SPB56/73689
1,cd221cdb-4df3-4939-9939-b550addeec6b,98b97815-a683-4c1d-9386-7395fbaffb2d,AC/SPB1167/1062
2,673d6cb5-7316-4a43-b704-7d71ff9e6673,c9d1ca03-3db7-4c2a-be7b-d84f19be615d,AC/SPB343/1035
3,9b619adf-338d-4968-841b-afda9182e689,d7a566c4-aa03-42ac-906a-aed9f86ae7a8,AC/SPB1084/810
4,2fd21da0-99d2-474d-aba8-bfdc3a6fea1b,dea1df91-449c-4d65-a74f-d90973de9bb5,AC/SPB1208/567
